<a href="https://colab.research.google.com/github/Tokhirjonov15/Menu_detector_AI/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
print('Hello World!')

Hello World!


In [11]:
from google.colab import drive
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from PIL import Image, UnidentifiedImageError
import os
import numpy as np

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_Path:', DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert',  # Label Grouping | Class Consolidation
    'cheesecake': 'dessert',      # Label Grouping | Class Consolidation
    'kebab': 'kebab',
    'pilaf': 'pilaf'
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Dataset_Path: /content/drive/MyDrive/food101_dataset
5
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


In [12]:
# Custom Dataset Class

class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    print('Images_Length', len(self.images))
    return len(self.images)

  def __getitem__(self, idx):
    img_path = self.images[idx]
    print('Image_Path', img_path)
    label = self.labels[idx]
    print('Label', label)
    try:
      image = Image.open(img_path).convert('RGB')
    except (UnidentifiedImageError, OSError):
      print(f"Skipping broken image: {img_path}")
      return self.__getitem__((idx + 1) % len(self.images))
    if self.transform:
      image = self.transform(image)
    return image, label

In [13]:
# Gather and Split Data

all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
  class_path = os.path.join(DATASET_PATH, original_class)
  print('Class_Path', class_path)
  if not os.path.exists(class_path):
    print(f"Warning: {class_path} not found")
    continue
  for img in os.listdir(class_path):
    if not img.endswith(('.jpg', '.jpeg', '.png')):
      full_path = os.path.join(class_path, img)
      all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

print("All_Images:", all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

Class_Path /content/drive/MyDrive/food101_dataset/hamburger
Class_Path /content/drive/MyDrive/food101_dataset/hot_dog
Class_Path /content/drive/MyDrive/food101_dataset/chocolate_cake
Class_Path /content/drive/MyDrive/food101_dataset/cheesecake
Class_Path /content/drive/MyDrive/food101_dataset/kebab
Class_Path /content/drive/MyDrive/food101_dataset/pilaf
All_Images: [('/content/drive/MyDrive/food101_dataset/kebab/Image_58.webp', 3), ('/content/drive/MyDrive/food101_dataset/kebab/Image_46.webp', 3), ('/content/drive/MyDrive/food101_dataset/kebab/Image_39.webp', 3)]
Images_Length 2
2
Image_Path /content/drive/MyDrive/food101_dataset/kebab/Image_58.webp
Label 3
